# 1. Data Understanding


In [ ]:
import kagglehub


path = kagglehub.dataset_download("sujay1844/used-car-prices")

print("Path to dataset files:", path)

In [ ]:
import matplotlib.pyplot as plt # visualization
import numpy as np # numerical calcu
import pandas as pd # table operations
import seaborn as sns # rich visualization with mathplit
import warnings # avoid warnings
from scipy.stats import norm # statistical data
warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv(f"{path}/train.csv")

In [ ]:
df

Inference: Loaded the data into pandas DataFrame

In [ ]:
df.shape

Inference:The dataset has 5847 observations and 14 columns

In [ ]:
df.columns

Inference: Understanding what the columns in the dataset represent

In [ ]:
df.sample(10)

Inference: Understanding 10 random sample observations in the data to understand that <br>
1)the main affecting factors can be model,year,kilometer_driver,fuel_type,owner_type,mileage,engine,power<br>
2)year is the year of manufactuing<br>
3)new_price is the showroom price<br>
4)price is the listing price

In [ ]:
col_list = list(df.columns)
for column in col_list:
    print(f'There are {df[column].nunique()} unique values in the column {column}')

Inference:
1) Unnamed: 0 column is the row id
2) The dataset contains 1804 Car Models
3) The dataset is taken from 11 locations
4) The dataset contains 22 years of data
5) The dataset contains 3 Fuel Types and 2 Transmission types
6) The dataset has 4 types of Owners

## Export for the next stage

Run this notebook first. It writes the dataset CSV that the Data Cleaning notebook reads.

In [ ]:
df.to_csv('usedcars_stage1_understood.csv', index=False)
print('Exported usedcars_stage1_understood.csv')


# 2. Two-stage Data Dictionary

The data dictionary is documented in two stages, as required for Phase 1:
- **Raw dictionary:** describes the source dataset before any cleaning.
- **Clean dictionary:** describes the intended final structure after the preparation steps documented in the supplied project data-dictionary PDF.

> **Scope note:** This uploaded notebook contains data loading and initial understanding/EDA. It does not contain the actual cleaning code. Therefore, the clean dictionary below documents the cleaning actions recorded in the supplied PDF; it does not claim that those transformations have already been executed in this notebook. Run the cleaning workflow before treating the clean schema as the schema of an exported cleaned dataset.

## 2.1 Raw (original) data dictionary

| Original column | Meaning | Raw type / role | Notes |
|---|---|---|---|
| Unnamed: 0 | Source row identifier | Integer / identifier | Row index or ID; not a vehicle feature. |
| Name | Car make and model | Categorical text | Brand and model as listed. |
| Location | Listing city | Categorical text | City name. |
| Year | Manufacturing year | Integer / numerical | Calendar year. |
| Kilometers_Driven | Distance travelled | Numerical | Kilometres. |
| Fuel_Type | Fuel category | Categorical text | Diesel, Petrol, Electric, etc. |
| Transmission | Transmission system | Categorical text | Manual, Automatic. |
| Owner_Type | Ownership category | Categorical text | First, Second, Third, Fourth & Above. |
| Mileage | Fuel-efficiency figure | Unit-bearing text | Verify raw suffixes before assigning one universal unit. |
| Engine | Engine capacity | Unit-bearing text | CC. |
| Power | Engine power | Unit-bearing text | bhp. |
| Seats | Number of seats | Numerical / integer-valued | Integer-valued, with possible missing values. |
| New_Price | New/showroom price | Text / object | 5,032 missing values out of 5,847. |
| Price | Used-car listing price | Numerical / target | Plots label it in lakh; exact source unit is not confirmed in the notes. |

## 2.2 Clean (final) data dictionary

| Final column | Meaning | Clean type / role | Preparation / notes |
|---|---|---|---|
| Name | Car make and model | String / categorical | Converted to string. |
| Location | Listing city | String / categorical | Converted to string. |
| Manufacturing_Year | Manufacturing year | Integer / numerical | Renamed from `Year`. |
| Kilometers_Driven | Distance travelled | Numerical | 6,500,000 treated as a likely data-entry error and replaced with the median for the same car Name. |
| Fuel_Type | Fuel category | String / categorical | Converted to string. |
| Transmission | Transmission system | String / categorical | Converted to string. |
| Owner_Type | Ownership category | String / categorical | Converted to string. |
| Mileage | Fuel-efficiency figure | Float / numerical | Numeric portion extracted. 40 zero values changed to missing; missing values filled by Name median, then Fuel_Type median. Electric vehicles may remain missing because kmpl is not applicable. |
| Engine | Engine capacity (CC) | Float / numerical | Numeric portion extracted; for non-electric vehicles, missing values filled by Name median, then Fuel_Type median. |
| Power | Engine power (bhp) | Float / numerical | Numeric portion extracted; for non-electric vehicles, missing values filled by Name median, then Fuel_Type median. |
| Seats | Number of seats | Nullable integer | Missing values filled by Name mode, then overall mode. |
| Price | Used-car resale/listing price | Numerical / target | Retained. 679 observations flagged by the 1.5 × IQR rule; not automatically removed. |

## 2.3 Column-level changes

- `Unnamed: 0` is dropped because it is only a row identifier.
- `Year` is renamed to `Manufacturing_Year`.
- `New_Price` is renamed to `Showroom_Price` and then dropped because 5,032 of 5,847 values are missing.
- `Name`, `Location`, `Fuel_Type`, `Transmission`, and `Owner_Type` are converted to string type.
- Numeric portions are extracted from `Mileage`, `Engine`, and `Power`, then converted to float.
- `Seats` is converted to nullable integer type.
- `Price` remains the target variable.

## 2.4 Data-quality observations and caveats

- The supplied notebook reports no duplicate rows.
- 40 zero values in `Mileage` are treated as missing before imputation.
- Mileage, Engine, and Power use grouped median imputation as described above. Electric-vehicle Mileage may remain missing because kmpl is not applicable.
- Missing Seats values use the mode for the same Name, followed by the overall mode.
- One extreme `Kilometers_Driven` value (6,500,000) is treated as a likely data-entry error and replaced with the median for the same car Name.
- 679 `Price` observations are flagged as outliers by the 1.5 × IQR rule. Flagging does not mean they are automatically deleted.
- Confirm the raw `Mileage` suffixes and the exact currency/unit for `Price` from source metadata before making stronger claims.

## 2.5 Dataset roles after preparation

- **Target / dependent variable:** `Price`
- **Potential features:** `Name`, `Location`, `Manufacturing_Year`, `Kilometers_Driven`, `Fuel_Type`, `Transmission`, `Owner_Type`, `Mileage`, `Engine`, `Power`, `Seats`
- **Removed identifier:** `Unnamed: 0`
- **Dropped high-missingness field:** `New_Price` (renamed to `Showroom_Price` before dropping)

## 2.6 Exportable dictionary tables

The next cell creates two CSV data-dictionary files from the documented definitions. The clean dictionary describes the recorded preparation plan and should be checked against the executed cleaning output.


In [ ]:

raw_dictionary = pd.DataFrame([
    {"column": "Unnamed: 0", "meaning": "Source row identifier", "raw_type_role": "Integer / identifier", "notes": "Row index or ID; not a vehicle feature."},
    {"column": "Name", "meaning": "Car make and model", "raw_type_role": "Categorical text", "notes": "Brand and model as listed."},
    {"column": "Location", "meaning": "Listing city", "raw_type_role": "Categorical text", "notes": "City name."},
    {"column": "Year", "meaning": "Manufacturing year", "raw_type_role": "Integer / numerical", "notes": "Calendar year."},
    {"column": "Kilometers_Driven", "meaning": "Distance travelled", "raw_type_role": "Numerical", "notes": "Kilometres."},
    {"column": "Fuel_Type", "meaning": "Fuel category", "raw_type_role": "Categorical text", "notes": "Examples: Diesel, Petrol, Electric."},
    {"column": "Transmission", "meaning": "Transmission system", "raw_type_role": "Categorical text", "notes": "Examples: Manual, Automatic."},
    {"column": "Owner_Type", "meaning": "Ownership category", "raw_type_role": "Categorical text", "notes": "First, Second, Third, Fourth & Above."},
    {"column": "Mileage", "meaning": "Fuel-efficiency figure", "raw_type_role": "Unit-bearing text", "notes": "Verify raw suffixes before assigning one universal unit."},
    {"column": "Engine", "meaning": "Engine capacity", "raw_type_role": "Unit-bearing text", "notes": "CC."},
    {"column": "Power", "meaning": "Engine power", "raw_type_role": "Unit-bearing text", "notes": "bhp."},
    {"column": "Seats", "meaning": "Number of seats", "raw_type_role": "Numerical / integer-valued", "notes": "Integer-valued; possible missing values."},
    {"column": "New_Price", "meaning": "New/showroom price", "raw_type_role": "Text / object", "notes": "5,032 missing values out of 5,847."},
    {"column": "Price", "meaning": "Used-car listing price", "raw_type_role": "Numerical / target", "notes": "Plots label it in lakh; exact source unit is not confirmed."}
])
clean_dictionary = pd.DataFrame([
    {"column": "Name", "meaning": "Car make and model", "clean_type_role": "String / categorical", "preparation": "Converted to string."},
    {"column": "Location", "meaning": "Listing city", "clean_type_role": "String / categorical", "preparation": "Converted to string."},
    {"column": "Manufacturing_Year", "meaning": "Manufacturing year", "clean_type_role": "Integer / numerical", "preparation": "Renamed from Year."},
    {"column": "Kilometers_Driven", "meaning": "Distance travelled", "clean_type_role": "Numerical", "preparation": "6,500,000 treated as likely data-entry error; replaced with median for same car Name."},
    {"column": "Fuel_Type", "meaning": "Fuel category", "clean_type_role": "String / categorical", "preparation": "Converted to string."},
    {"column": "Transmission", "meaning": "Transmission system", "clean_type_role": "String / categorical", "preparation": "Converted to string."},
    {"column": "Owner_Type", "meaning": "Ownership category", "clean_type_role": "String / categorical", "preparation": "Converted to string."},
    {"column": "Mileage", "meaning": "Fuel-efficiency figure", "clean_type_role": "Float / numerical", "preparation": "Numeric portion extracted; 40 zeros changed to missing; imputed by Name median then Fuel_Type median. EVs may remain missing because kmpl is not applicable."},
    {"column": "Engine", "meaning": "Engine capacity (CC)", "clean_type_role": "Float / numerical", "preparation": "Numeric portion extracted; non-EV missing values imputed by Name median then Fuel_Type median."},
    {"column": "Power", "meaning": "Engine power (bhp)", "clean_type_role": "Float / numerical", "preparation": "Numeric portion extracted; non-EV missing values imputed by Name median then Fuel_Type median."},
    {"column": "Seats", "meaning": "Number of seats", "clean_type_role": "Nullable integer", "preparation": "Missing values imputed by Name mode then overall mode."},
    {"column": "Price", "meaning": "Used-car resale/listing price", "clean_type_role": "Numerical / target", "preparation": "Retained; 679 outliers flagged by 1.5 × IQR, not automatically removed."}
])
raw_dictionary.to_csv("raw_data_dictionary.csv", index=False)
clean_dictionary.to_csv("clean_data_dictionary.csv", index=False)
print("Exported raw_data_dictionary.csv and clean_data_dictionary.csv")
